<a href="https://colab.research.google.com/github/Don4141/ProductPriceAI/blob/main/Traditional_ML_Modeling_and_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The goal here is to develop and evaluate machine learning and deep learning models for product price prediction using the full preprocessed dataset hosted on Hugging Face, rather than the lite version. Baseline and other traditional machine learning models are trained and evaluated to establish reference performance.

Deep learning models, including an Artificial Neural Network (ANN) and a Stacked Autoencoder (SAE)-based regression model, are implemented to explore complex relationships between product summaries and their corresponding prices. Model performance is evaluated and compared to assess predictive accuracy and the effectiveness of the different modeling methods.

Mount google drive

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Verify Colab can retrieve your GitHub secret.

In [3]:
from google.colab import userdata
github_token = userdata.get("GITHUB_TOKEN")
print("Github token added", github_token is not None)

Github token added True


In [4]:
import os
os.chdir("/content")
#Clone repo after a new Colab runtime as the previous will be erased
!git clone https://github.com/Don4141/ProductPriceAI.git

#Set repository as working directory
os.chdir("/content/ProductPriceAI")

#Verify
print("Repo exist", os.path.exists("/content/ProductPriceAI"))
print("Current working directory", os.getcwd())

Cloning into 'ProductPriceAI'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 130 (delta 55), reused 79 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (130/130), 14.12 MiB | 21.35 MiB/s, done.
Resolving deltas: 100% (55/55), done.
Repo exist True
Current working directory /content/ProductPriceAI


In [ ]:
# Load API credentials
for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GROQ_API_KEY"]:
  os.environ[key] = userdata.get(key)

In [ ]:
 #Verify if API keys are loaded
for key in ["OPENAI_API_KEY", "ANTHROPIC_API_KEY", "GROQ_API_KEY"]:
    print(f"{key}: {'loaded' if os.getenv(key) else 'Missing'}")

OPENAI_API_KEY: loaded
ANTHROPIC_API_KEY: loaded
GROQ_API_KEY: loaded


In [ ]:
#Import project modules
import random
import pandas as pd
import numpy as np
import stat
import copy
import math
from pricer.items import Item
from dotenv import load_dotenv
from huggingface_hub import login
import xgboost as xgb
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from torch.utils.data import Dataset, DataLoader, DataLoader, TensorDataset
from torch.optim.lr_scheduler import CosineAnnealingLR
import torch.nn as nn
import torch.optim as optim
#from pricer.evaluator import evaluate, Tester
#from litellm import completion
from tqdm.auto import tqdm
from tqdm.notebook import tqdm
import csv
#import torch

Load the preprocessed dataset containing 819,998 products from Hugging Face

In [ ]:
dataset = f"{"sadjei65320"}/items_full"
train, val, test = Item.from_hub(dataset)
items = train + val + test

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

README.md:   0%|          | 0.00/744 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  237MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.00MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.00MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/799998 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loaded 799,998 training items, 10,000 validation items, 10,000 test items


Establish a reference against which the trained models can be compared

In [ ]:
#Define a random baseline that predicts product prices between $0 and $999.
#This baseline model just takes an item and estimate its price without considering any product features in the prediction.
def random_pricer(item):
    return random.randrange(1,1000)

In [ ]:
#Evaluate the performance of the function on the first 1000 items in the test data.
#The evaluate() function iterates through the test products, obtains predictions for 1000 of the products, and compares them with their actual prices.
random.seed(42)
random_pricer_tester = evaluate(random_pricer, test, size=1000)

  0%|          | 0/1000 [00:00<?, ?it/s]

$596 $6 $263 $688 $104 $687 $193 $668 $104 $119 $414 $195 $868 $52 $184 $590 $169 $108 $680 $664 $268 $542 $89 $60 $24 $190 $494 $499 $8 $540 $124 $684 $641 $589 $411 $20 $0 $86 $304 $145 $699 $428 $226 $743 $781 $83 $655 $386 $174 $258 $139 $208 $501 $629 $84 $85 $25 $360 $82 $129 $654 $266 $537 $227 $795 $36 $721 $451 $470 $78 $982 $761 $338 $46 $326 $272 $745 $628 $606 $867 $865 $291 $471 $7 $582 $28 $32 $588 $1 $746 $280 $47 $834 $215 $828 $49 $90 $215 $405 $591 $844 $185 $57 $296 $184 $112 $672 $66 $563 $861 $620 $574 $38 $568 $91 $361 $497 $593 $154 $140 $334 $86 $253 $838 $516 $662 $512 $552 $771 $320 $171 $450 $513 $2 $182 $815 $57 $800 $122 $226 $145 $127 $120 $853 $936 $548 $875 $650 $188 $200 $642 $392 $306 $848 $837 $549 $375 $123 $244 $118 $326 $747 $427 $533 $119 $564 $534 $399 $807 $572 $347 $352 $85 $27 $444 $205 $26 $674 $141 $826 $127 $137 $175 $138 $792 $452 $378 $383 $145 $362 $268 $571 $463 $382 $234 $896 $518 $773 $936 $16 $632 $394 $106 $669 $582 $429 $689 $75 $6

In [ ]:
random_pricer_tester.report()

In [ ]:
#Let's build another trivial model
training_prices = [item.price for item in train]
training_average = sum(training_prices) / len(training_prices)
print(training_average)

143.9564089160223


In [ ]:
def constant_pricer(item):
    return training_average

In [ ]:
constant_pricer_tester = evaluate(constant_pricer, test, size=1000)

  0%|          | 0/1000 [00:00<?, ?it/s]

$85 $124 $234 $72 $3 $825 $108 $57 $65 $120 $135 $220 $98 $106 $431 $129 $88 $81 $122 $56 $21 $431 $24 $108 $56 $285 $120 $25 $108 $109 $64 $94 $119 $14 $5 $306 $82 $230 $156 $4 $14 $319 $89 $109 $99 $103 $84 $97 $31 $117 $123 $131 $336 $9 $117 $124 $74 $114 $126 $95 $70 $57 $62 $100 $112 $63 $117 $124 $64 $94 $128 $40 $94 $109 $96 $115 $39 $128 $116 $104 $126 $64 $23 $46 $4 $100 $129 $54 $91 $98 $127 $109 $102 $120 $84 $89 $336 $74 $84 $84 $132 $45 $34 $60 $36 $41 $129 $64 $13 $45 $64 $54 $108 $88 $416 $393 $94 $10 $261 $116 $4 $331 $120 $34 $4 $101 $85 $216 $128 $129 $31 $128 $138 $88 $91 $117 $54 $119 $58 $41 $14 $51 $47 $61 $114 $111 $121 $58 $9 $126 $114 $24 $44 $86 $44 $34 $49 $126 $116 $119 $435 $128 $4 $125 $7 $57 $79 $104 $31 $118 $82 $125 $4 $25 $66 $157 $76 $44 $46 $88 $96 $124 $674 $118 $124 $101 $89 $84 $67 $111 $21 $104 $127 $16 $120 $45 $95 $35 $114 $116 $79 $201 $132 $114 $181 $23 $64 $55 $28 $113 $84 $98 $44 $6 $126 $133 $130 $123 $118 $124 $95 $129 $295 $114 $105 $94 

In [ ]:
constant_pricer_tester.report()

In [ ]:
def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight==0 else 0,
        "text_length": len(item.summary)
    }

In [ ]:
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

In [ ]:
#Traditional Linear Regression!
np.random.seed(42)
# Separate features and target
feature_columns = ['weight', 'weight_unknown', 'text_length']

#Continuous features to scale
num_vars = ['weight', 'text_length']#Excluding weight_unknown because it's a binary indicator (o or 1)
X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

#Scale continuous features while retaining weight_unknown unchanged
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_vars)
    ],
    remainder='passthrough'
)

#Create linear regression pipeline
model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

# Fit and train the Linear Regression
model.fit(X_train, y_train)
#Get transformed feature names
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
#Access the fitted linear regression estimator
regressor = model.named_steps['regressor']

# Print the coefficients and intercept
for feature, coef in zip(feature_names, regressor.coef_):
    print(f"\n{feature}: {coef:.4}")
print(f"\nIntercept: {regressor.intercept_:.4}")

# Predict the test set and evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"\nMean Squared Error: {mse:.4f}")
print(f"\nR-squared Score: {r2:.4f}")


num__weight: 19.8

num__text_length: 13.35

remainder__weight_unknown: -7.155

Intercept: 144.9

Mean Squared Error: 24373.6703

R-squared Score: 0.0356


Let's try to understand the coefficients associated with the features and the intercept of the linear regression.

The intercept is the predicted price when both weight and text length features are zero and weight_unknown is zero.

Since weight and text_length are standardized, their coefficients are not interpreted per original unit. The weight_unknown coefficient, however, remains associated with a change from 0 to 1.

The coefficient associated with weight(scaled) is interpretted as follows: A one-standard-deviation increase in weight is associated with a \$19.80 increase in predicted price, holding text length and weight unknow features constant.

For the text_length (scaled) coefficient of \+13.35:	A one-standard-deviation increase in text length is associated with a \$13.35 increase in predicted price, holding the coeffients of weight and weight unknow constant.

For weight_unknown coefficient of \-7.16:	Missing weight is associated with a \$7.16 lower predicted price, holding the coefficients of weight and text length constant.

These coefficients suggest that both product weight and text length are positively associated with price in the training data while a missing weight is associated with a decrease a product price.

However, these coefficients do not establish causation, and their values alone do not tell us how accurately the model predicts product prices.

In [ ]:
def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])[feature_columns]
    return model.predict(features_df)[0]

In [ ]:
item = test[0]
predicted_price = linear_regression_pricer(item)

print(f"Predicted price: ${predicted_price:,.2f}")
print(f"Actual price: ${item.price:,.2f}")

Predicted price: $133.06
Actual price: $58.73


In [ ]:
linear_regression_pricer_tester = evaluate(linear_regression_pricer, test, size=1000)

  0%|          | 0/1000 [00:00<?, ?it/s]

$74 $117 $236 $75 $5 $825 $102 $55 $31 $120 $124 $174 $84 $90 $400 $114 $89 $76 $135 $33 $40 $410 $28 $84 $47 $293 $110 $44 $122 $97 $62 $89 $128 $19 $29 $259 $71 $219 $144 $21 $17 $294 $63 $105 $101 $82 $72 $79 $39 $130 $131 $160 $348 $11 $135 $118 $55 $122 $126 $58 $80 $57 $57 $125 $112 $53 $115 $125 $70 $84 $137 $21 $77 $100 $111 $132 $38 $101 $122 $99 $102 $74 $28 $51 $17 $100 $128 $63 $82 $117 $123 $108 $86 $121 $93 $86 $343 $74 $97 $78 $138 $45 $17 $99 $38 $22 $113 $97 $14 $52 $58 $44 $77 $84 $405 $373 $113 $6 $265 $96 $2 $317 $113 $20 $7 $90 $76 $221 $122 $122 $34 $129 $124 $73 $103 $132 $74 $121 $48 $70 $26 $51 $23 $48 $101 $104 $128 $72 $5 $121 $110 $40 $50 $91 $56 $56 $58 $121 $118 $124 $427 $129 $11 $112 $26 $74 $66 $113 $26 $94 $71 $130 $10 $12 $79 $144 $92 $55 $52 $84 $105 $137 $667 $106 $120 $97 $83 $94 $66 $107 $1 $110 $110 $2 $126 $41 $80 $40 $121 $130 $77 $220 $129 $104 $192 $2 $79 $52 $33 $99 $96 $97 $40 $8 $124 $134 $126 $125 $119 $142 $95 $129 $273 $95 $126 $94 $102

In [ ]:
linear_regression_pricer_tester.report()

In [ ]:
#Building a baseline for predicting product prices from their natural language descriptions
#Words in each product's summary is used as an input feature
prices = np.array([float(item.price) for item in train])#Extract product prices
documents = [item.summary for item in train]#Extract product summaries

In [ ]:
np.random.seed(42)
#Create text regression pipeline
nlp_model = Pipeline([
    ('vectorizer', CountVectorizer(max_features=2000, stop_words='english')),#Converts text into numerical features based on word frequencies. Limits each product summary to 2000 words
    ('regressor', LinearRegression())])#Regression algorithm finds coefficients that minimize the sum of squared prediction errors.

nlp_model.fit(documents, prices)#Transforms summaries into numeric features

Pipeline(steps=[('vectorizer',
                 CountVectorizer(max_features=2000, stop_words='english')),
                ('regressor', LinearRegression())])

In [ ]:
#Retrieve the vocabulary learned during training
selected_words = nlp_model.named_steps['vectorizer'].get_feature_names_out()
#selected_words = vectorizer.get_feature_names_out()
print(f"Number of selected words: {len(selected_words)}")
print("Selected words:", selected_words[1000:1020])

Number of selected words: 2000
Selected words: ['jack' 'jacket' 'jeep' 'jet' 'jigsaw' 'joint' 'joints' 'kawasaki'
 'keeping' 'keeps' 'key' 'keyboard' 'keypad' 'keys' 'kg' 'khz' 'kia'
 'kickstand' 'kids' 'king']


In [ ]:
def natural_language_linear_regression_pricer(item):
    pred = nlp_model.predict([item.summary])[0]#Transforms new product summaries into numerical vector
    return max(float(pred), 0)#Generates prediction using trained regression model and ensures no returned price for a product is negative.

In [ ]:
natural_language_linear_regression_pricer_tester = evaluate(natural_language_linear_regression_pricer, test, size=1000)

  0%|          | 0/1000 [00:00<?, ?it/s]

$11 $75 $147 $1 $24 $810 $14 $133 $8 $94 $58 $207 $97 $20 $260 $4 $17 $8 $22 $17 $91 $341 $111 $31 $45 $315 $115 $56 $17 $104 $51 $6 $42 $138 $9 $73 $46 $124 $5 $37 $151 $221 $107 $159 $30 $50 $68 $46 $35 $27 $34 $4 $191 $72 $48 $20 $35 $90 $18 $53 $93 $6 $106 $85 $32 $1 $88 $13 $102 $82 $8 $190 $3 $35 $8 $176 $122 $5 $70 $192 $56 $164 $40 $20 $108 $48 $36 $45 $46 $53 $4 $153 $16 $6 $52 $43 $281 $55 $17 $18 $86 $38 $109 $59 $2 $66 $15 $25 $59 $45 $200 $23 $28 $115 $210 $323 $21 $41 $5 $18 $75 $378 $90 $131 $64 $6 $134 $151 $6 $4 $109 $0 $35 $37 $71 $45 $44 $26 $49 $52 $62 $72 $41 $10 $7 $43 $132 $70 $17 $78 $28 $11 $24 $150 $193 $53 $111 $15 $35 $25 $41 $50 $207 $12 $77 $60 $126 $22 $29 $26 $164 $19 $137 $32 $19 $145 $95 $28 $37 $74 $211 $2 $438 $132 $12 $24 $80 $52 $90 $20 $22 $63 $17 $2 $140 $90 $159 $20 $74 $28 $102 $145 $12 $20 $100 $35 $89 $33 $13 $44 $50 $135 $14 $45 $17 $42 $141 $207 $74 $19 $7 $4 $348 $22 $119 $87 $45 $23 $63 $13 $38 $103 $84 $147 $13 $77 $100 $154 $30 $5 $124 

In [ ]:
natural_language_linear_regression_pricer_tester.report()

In [ ]:
#Train random forest model
subset = 100_000
rf_model = RandomForestRegressor(n_estimators=100,
                       n_jobs=-1, random_state=42)

vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)
rf_model.fit(X[:subset], prices[:subset])

RandomForestRegressor(n_jobs=-1, random_state=42)

In [ ]:
def random_forest_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

In [ ]:
random_forest_pricer_tester = evaluate(random_forest_pricer, test, size=1000)

  0%|          | 0/1000 [00:00<?, ?it/s]

$9 $37 $188 $7 $23 $805 $2 $53 $24 $157 $47 $179 $19 $12 $354 $26 $15 $15 $26 $23 $72 $338 $68 $4 $51 $337 $25 $88 $32 $66 $44 $16 $60 $26 $10 $200 $45 $86 $20 $12 $195 $179 $3 $81 $4 $132 $24 $55 $55 $11 $65 $3 $125 $74 $87 $9 $31 $63 $3 $74 $75 $63 $70 $183 $10 $9 $309 $15 $30 $54 $187 $63 $12 $34 $87 $236 $31 $9 $46 $95 $47 $50 $39 $2 $108 $126 $51 $36 $29 $23 $1 $77 $8 $10 $5 $3 $331 $33 $29 $3 $29 $143 $63 $27 $9 $43 $33 $69 $43 $56 $46 $22 $11 $106 $263 $329 $10 $74 $48 $85 $8 $387 $40 $172 $67 $5 $76 $177 $38 $35 $100 $14 $19 $31 $38 $8 $48 $3 $9 $93 $43 $49 $47 $4 $2 $9 $35 $1 $37 $44 $1 $9 $29 $34 $173 $1 $45 $79 $75 $3 $54 $10 $77 $8 $20 $17 $113 $56 $20 $33 $99 $6 $105 $27 $78 $153 $98 $3 $64 $27 $146 $14 $323 $178 $65 $11 $48 $23 $80 $1 $47 $17 $1 $17 $144 $14 $101 $40 $44 $6 $38 $279 $97 $93 $112 $55 $37 $48 $51 $8 $28 $38 $50 $40 $25 $15 $84 $171 $79 $14 $2 $1 $365 $40 $146 $35 $27 $42 $61 $8 $25 $233 $94 $86 $23 $18 $20 $90 $142 $21 $201 $71 $17 $85 $4 $0 $33 $50 $92 $3 

In [ ]:
random_forest_pricer_tester.report()

In [ ]:
np.random.seed(42)
xgb_model = xgb.XGBRegressor(n_estimators=1000, max_depth=6, learning_rate=0.1,
                              tree_method="hist",random_state=42, n_jobs=4)
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)
xgb_model.fit(X, prices)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=4, num_parallel_tree=None, ...)

In [ ]:
def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

In [ ]:
xg_boost_tester = evaluate(xg_boost, test, size=1000)

  0%|          | 0/1000 [00:00<?, ?it/s]

$6 $91 $204 $1 $23 $697 $21 $106 $28 $132 $74 $191 $50 $6 $254 $25 $3 $61 $9 $63 $75 $226 $101 $18 $50 $270 $98 $4 $12 $61 $80 $12 $25 $78 $5 $121 $40 $67 $59 $19 $140 $239 $48 $133 $7 $41 $43 $74 $26 $3 $69 $1 $47 $22 $4 $5 $21 $45 $11 $67 $105 $36 $91 $87 $24 $5 $130 $4 $65 $43 $5 $84 $2 $26 $49 $202 $115 $18 $51 $246 $71 $84 $6 $44 $46 $57 $42 $14 $119 $96 $26 $135 $9 $39 $8 $33 $305 $42 $35 $27 $44 $41 $59 $32 $6 $21 $30 $26 $18 $67 $165 $32 $34 $128 $141 $264 $12 $68 $2 $34 $24 $332 $73 $53 $34 $9 $57 $146 $38 $2 $84 $21 $38 $60 $23 $67 $55 $10 $62 $104 $63 $24 $3 $18 $23 $29 $74 $24 $9 $52 $8 $10 $19 $91 $183 $0 $106 $42 $61 $2 $78 $46 $88 $3 $4 $31 $83 $30 $39 $4 $125 $1 $98 $1 $17 $82 $116 $10 $45 $63 $173 $20 $220 $172 $5 $100 $50 $7 $79 $4 $6 $62 $10 $32 $106 $44 $95 $22 $66 $22 $72 $109 $12 $53 $96 $43 $52 $14 $2 $37 $58 $61 $22 $34 $18 $27 $117 $140 $106 $26 $12 $24 $372 $24 $145 $47 $26 $37 $63 $19 $38 $143 $89 $99 $23 $2 $87 $133 $105 $15 $116 $78 $29 $70 $14 $31 $6 $25 $

In [ ]:
xg_boost_tester.report()

Neural Network

In [ ]:
# Prepare our documents and prices
y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [ ]:
# Use the HashingVectorizer for a Bag of Words model
# Using binary=True with the CountVectorizer makes "one-hot vectors"
np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [ ]:
# Define the neural network 4 layers
class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 256)
        self.layer2 = nn.Linear(256, 128)
        self.layer3 = nn.Linear(128, 64)
        self.layer4 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.layer4(output3)
        return output4

In [ ]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 1,325,633


In [ ]:
# Define loss function and optimizer
loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
# We will do 2 complete runs through the data
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    #Validate the trained model on the unseen data
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f"Epoch {epoch+1}/{EPOCHS} | " f"Train Loss: {loss.item():.3f} | " f"Val Loss: {val_loss.item():.3f}")

  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 8673.834, Val Loss: 12778.624


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 11887.243, Val Loss: 11682.790


In [ ]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [ ]:
neural_network_tester = evaluate(neural_network, test, size=1000)

  0%|          | 0/1000 [00:00<?, ?it/s]

$24 $69 $185 $75 $14 $790 $6 $36 $57 $149 $101 $140 $29 $3 $260 $22 $11 $77 $14 $36 $43 $28 $146 $33 $70 $87 $39 $31 $15 $51 $53 $6 $14 $64 $99 $90 $143 $11 $259 $32 $23 $196 $4 $93 $5 $32 $93 $28 $26 $12 $20 $2 $134 $43 $27 $13 $1 $3 $5 $77 $53 $46 $89 $39 $8 $19 $184 $5 $2 $43 $23 $37 $11 $22 $137 $89 $77 $11 $56 $95 $33 $65 $102 $50 $149 $50 $47 $1 $34 $36 $24 $44 $10 $6 $15 $16 $56 $38 $32 $23 $34 $2 $65 $45 $16 $31 $24 $53 $27 $41 $15 $28 $26 $76 $71 $334 $11 $29 $87 $54 $60 $318 $57 $62 $25 $2 $49 $154 $26 $10 $105 $17 $13 $14 $31 $22 $55 $5 $4 $115 $48 $90 $22 $45 $5 $4 $46 $51 $6 $29 $7 $33 $13 $158 $128 $6 $65 $80 $101 $1 $27 $17 $55 $0 $12 $5 $40 $11 $48 $60 $198 $1 $14 $0 $15 $172 $69 $22 $52 $115 $157 $5 $189 $75 $18 $34 $7 $113 $28 $32 $31 $61 $6 $34 $29 $6 $48 $93 $28 $6 $31 $80 $15 $18 $105 $38 $40 $139 $29 $16 $25 $50 $48 $61 $85 $21 $67 $133 $74 $15 $15 $6 $320 $29 $114 $20 $10 $15 $44 $18 $4 $115 $65 $99 $14 $4 $15 $75 $107 $27 $158 $82 $49 $97 $10 $17 $22 $6 $76 $31 

In [ ]:
neural_network_tester.report()

In [ ]:
#Prepare the dataset
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#Prepare documents and prices
documents = [item.summary or "" for item in train]
y = np.array([float(item.price) for item in train])

#Use the HashingVectorizer for a Bag of Words model
vectorizer = HashingVectorizer(n_features=5000, stop_words="english", binary=True)#Binary=True creates binary term presence features
X = vectorizer.transform(documents)#Converts texts in documents into numeric representations for the machine learning process.
                                  #Each column feature is one-hot encode into 0s and 1s

#Split the dataset into training and validation data
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.01, random_state=42)
#Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

#Create batches of hashed features and prices for the SAE pipeline
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"Device: {device}")
print(f"Training samples: {X_train.shape[0]:,}")
print(f"Validation samples: {X_val.shape[0]:,}")
print(f"Input features: {X_train.shape[1]:,}"

Device: cuda
Training samples: 791,998
Validation samples: 8,000
Input features: 5,000


In [ ]:
#Define the stacked autoencoder (SAE) to reconstruct the 5000 dimensional product text features
class SAE(nn.Module):
    def __init__(self, input_size):#Specifies the number of input size which in this case is 5000 numeric features
        super().__init__()
        #Define the encoder
        self.encoder = nn.Sequential(nn.Linear(input_size, 256), nn.ReLU(),
                                    nn.Linear(256, 128), nn.ReLU(),
                                    nn.Linear(128, 64), nn.ReLU())# Extract 64 dimensional encoded representations
        #Define the decoder
        self.decoder = nn.Sequential(nn.Linear(64, 128), nn.ReLU(),
                                    nn.Linear(128, 256), nn.ReLU(),
                                    nn.Linear(256, input_size))
    #Connect the eNcoder and the decoder
    def forward(self, x):
        return self.decoder(self.encoder(x))

In [ ]:
#Pretrain the autoencoder
# Initialize the model
auto_encoder = SAE(input_size=X_train_tensor.shape[1]).to(device)
SAE_loss = nn.MSELoss()
optimizer = optim.Adam(auto_encoder.parameters(), lr=0.001)

EPOCHS = 5
for epoch in range(EPOCHS):
    auto_encoder.train()
    train_loss = 0.0

    for batch_X, _ in tqdm(train_loader, desc=f"SAE Epoch {epoch+1}"):
        batch_X = batch_X.to(device)
        optimizer.zero_grad()
        decoded = auto_encoder(batch_X)
        loss = SAE_loss(decoded, batch_X)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch_X.size(0)#Training loss for the current batch
    train_loss /= len(train_loader.dataset)#Calculates the average loss per product across the entire batch

    # Validation
    auto_encoder.eval()
    val_loss = 0.0

    with torch.no_grad():
        for batch_X, _ in val_loader:
            batch_X = batch_X.to(device)
            decoded = auto_encoder(batch_X)
            loss = SAE_loss(decoded, batch_X)
            val_loss += loss.item() * batch_X.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{EPOCHS} | " f"Train SAE MSE: {train_loss:.6f} | " f"Val SAE MSE: {val_loss:.6f}")

SAE Epoch 1:   0%|          | 0/12375 [00:00<?, ?it/s]

Epoch 1/5 | Train Reconstruction MSE: 0.000142 | Val Reconstruction MSE: 0.000135


SAE Epoch 2:   0%|          | 0/12375 [00:00<?, ?it/s]

Epoch 2/5 | Train Reconstruction MSE: 0.000132 | Val Reconstruction MSE: 0.000129


SAE Epoch 3:   0%|          | 0/12375 [00:00<?, ?it/s]

Epoch 3/5 | Train Reconstruction MSE: 0.000128 | Val Reconstruction MSE: 0.000127


SAE Epoch 4:   0%|          | 0/12375 [00:00<?, ?it/s]

Epoch 4/5 | Train Reconstruction MSE: 0.000126 | Val Reconstruction MSE: 0.000125


SAE Epoch 5:   0%|          | 0/12375 [00:00<?, ?it/s]

Epoch 5/5 | Train Reconstruction MSE: 0.000125 | Val Reconstruction MSE: 0.000124


In [ ]:
#Define the SAE regressor
class SAEPriceRegressor(nn.Module):#Regressor uses pretrained encoder and add regression head
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.regressor = nn.Sequential(nn.Linear(64, 32), nn.ReLU(),
                         nn.Linear(32, 1))

    def forward(self, x):
        features = self.encoder(x)
        return self.regressor(features).squeeze(-1)

In [ ]:
#Fine-tune the entire encoder and regression head
price_model = SAEPriceRegressor(auto_encoder.encoder).to(device)#auto_encoder.encoder contains the pretrained encoder weights
for param in price_model.parameters():#Unfreeze the encoder parameters
    param.requires_grad = True

loss_function = nn.MSELoss() #Define the loss function
optimizer = optim.Adam([#Define the optimizer parameters for both the encoder and regression head which will both be updated duringg supevised training
    {"params": price_model.encoder.parameters(), "lr": 0.0001},
    {"params": price_model.regressor.parameters(), "lr": 0.001}
])

In [ ]:
#Train and validate the fine-tuned model
EPOCHS = 20
best_val_loss = float("inf")
best_state = None

for epoch in range(EPOCHS):
    # Training
    price_model.train()
    train_loss = 0.0

    for batch_X, batch_y in tqdm(train_loader, desc=f"Price Epoch {epoch+1}"):
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        predictions = price_model(batch_X)
        loss = loss_function(predictions, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_X.size(0)
    train_loss /= len(train_loader.dataset)

    # Validation
    price_model.eval()
    val_loss = 0.0
    val_mae = 0.0

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            predictions = price_model(batch_X)
            loss = loss_function(predictions, batch_y)

            val_loss += loss.item() * batch_X.size(0)
            val_mae += torch.abs(predictions - batch_y).sum().item()
    val_loss /= len(val_loader.dataset)
    val_mae /= len(val_loader.dataset)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(price_model.state_dict())#Select best model based on validation MSE

    print(f"Epoch {epoch+1}/{EPOCHS} | " f"Train MSE: {train_loss:.2f} | " f"Val MSE: {val_loss:.2f} | " f"Val RMSE: ${math.sqrt(val_loss):.2f} | " f"Val MAE: ${val_mae:.2f}")

# Restore the best model
price_model.load_state_dict(best_state)
price_model.eval()

Price Epoch 1:   0%|          | 0/12375 [00:00<?, ?it/s]

Epoch 1/5 | Train MSE: 16683.69 | Val MSE: 15665.91 | Val RMSE: $125.16 | Val MAE: $76.96


Price Epoch 2:   0%|          | 0/12375 [00:00<?, ?it/s]

Epoch 2/5 | Train MSE: 13535.61 | Val MSE: 14135.57 | Val RMSE: $118.89 | Val MAE: $75.15


Price Epoch 3:   0%|          | 0/12375 [00:00<?, ?it/s]

Epoch 3/5 | Train MSE: 11795.86 | Val MSE: 13101.56 | Val RMSE: $114.46 | Val MAE: $71.28


Price Epoch 4:   0%|          | 0/12375 [00:00<?, ?it/s]

Epoch 4/5 | Train MSE: 10415.47 | Val MSE: 12514.27 | Val RMSE: $111.87 | Val MAE: $67.73


Price Epoch 5:   0%|          | 0/12375 [00:00<?, ?it/s]

Epoch 5/5 | Train MSE: 9279.19 | Val MSE: 12568.68 | Val RMSE: $112.11 | Val MAE: $69.40


SAEPriceRegressor(
  (encoder): Sequential(
    (0): Linear(in_features=5000, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
  )
  (regressor): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [ ]:
#Define the prediction function
def SAE_pricer(item):
    price_model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary or ""])
        vector = torch.FloatTensor(vector.toarray(), device=device)
        prediction = price_model(vector)[0].item()
    return max(0.0, prediction)

In [ ]:
#Evaluate on the selected model on the heald-out test set
SAE_pricer_tester = evaluate(SAE_pricer, test, size=1000)

  0%|          | 0/1000 [00:00<?, ?it/s]

$14 $81 $118 $44 $19 $756 $4 $49 $21 $117 $73 $137 $39 $3 $284 $24 $15 $105 $12 $88 $172 $73 $55 $22 $88 $255 $67 $34 $17 $109 $42 $22 $35 $44 $10 $139 $92 $36 $108 $1 $55 $149 $22 $63 $11 $86 $24 $71 $40 $11 $33 $1 $16 $68 $6 $16 $7 $2 $2 $81 $129 $40 $126 $76 $1 $0 $112 $12 $25 $80 $67 $12 $30 $13 $152 $89 $55 $4 $42 $97 $33 $53 $76 $85 $113 $29 $60 $16 $158 $35 $16 $86 $5 $18 $9 $21 $145 $41 $44 $33 $27 $11 $56 $45 $19 $18 $21 $99 $81 $41 $72 $78 $7 $87 $72 $359 $2 $51 $17 $26 $29 $327 $46 $67 $41 $9 $25 $151 $29 $27 $86 $15 $6 $4 $27 $26 $49 $3 $37 $121 $28 $69 $39 $42 $4 $7 $66 $40 $14 $14 $23 $15 $27 $112 $57 $3 $132 $82 $102 $0 $229 $15 $72 $2 $48 $122 $83 $28 $37 $40 $152 $9 $74 $34 $18 $131 $54 $51 $35 $157 $211 $7 $146 $84 $20 $12 $17 $106 $63 $47 $52 $40 $7 $3 $28 $52 $64 $45 $49 $6 $10 $13 $26 $34 $119 $42 $2 $18 $20 $6 $25 $53 $40 $49 $48 $13 $63 $120 $32 $57 $13 $1 $341 $31 $131 $17 $14 $59 $77 $13 $15 $112 $57 $108 $12 $3 $103 $61 $114 $54 $94 $76 $25 $58 $6 $25 $25 $10 

In [ ]:
SAE_pricer_tester.report()